In [1]:
import pandas as pd
import numpy as np
from google.colab import files

SIDO_NAMES = {
    '서울특별시','부산광역시','대구광역시','인천광역시','광주광역시','대전광역시','울산광역시',
    '세종특별자치시','경기도','강원특별자치도','충청북도','충청남도','전북특별자치도',
    '전라남도','경상북도','경상남도','제주특별자치도','전남광주통합특별시',
}
SIDO_ALIASES = {
    '강원도': '강원특별자치도',
    '전라북도': '전북특별자치도',
    '광주광역시': '전남광주통합특별시',   # 2026.7.1 광주·전남 통합
    '전라남도': '전남광주통합특별시',     # 2026.7.1 광주·전남 통합
}
NON_REGION = {'전국', '동부', '읍부', '면부'}
GU_HAVING_CITIES = {'수원시','성남시','안양시','안산시','고양시','용인시',
                     '청주시','천안시','전주시','포항시','창원시','부천시'}
BARE_GU_TO_CITY = {
    ('경기도','장안구'):'수원시', ('경기도','권선구'):'수원시', ('경기도','팔달구'):'수원시', ('경기도','영통구'):'수원시',
    ('경기도','수정구'):'성남시', ('경기도','중원구'):'성남시', ('경기도','분당구'):'성남시',
    ('경기도','만안구'):'안양시', ('경기도','동안구'):'안양시',
    ('경기도','상록구'):'안산시', ('경기도','단원구'):'안산시',
    ('경기도','덕양구'):'고양시', ('경기도','일산동구'):'고양시', ('경기도','일산서구'):'고양시',
    ('경기도','처인구'):'용인시', ('경기도','기흥구'):'용인시', ('경기도','수지구'):'용인시',
    ('경기도','원미구'):'부천시', ('경기도','소사구'):'부천시', ('경기도','오정구'):'부천시',
    ('충청북도','상당구'):'청주시', ('충청북도','서원구'):'청주시', ('충청북도','흥덕구'):'청주시', ('충청북도','청원구'):'청주시',
    ('충청남도','동남구'):'천안시', ('충청남도','서북구'):'천안시',
    ('전북특별자치도','완산구'):'전주시', ('전북특별자치도','덕진구'):'전주시',
    ('경상북도','남구'):'포항시', ('경상북도','북구'):'포항시',
    ('경상남도','의창구'):'창원시', ('경상남도','성산구'):'창원시',
    ('경상남도','마산합포구'):'창원시', ('경상남도','마산회원구'):'창원시', ('경상남도','진해구'):'창원시',
}
LEGACY_SIDO_OVERRIDE = {
    ('경상북도','군위군'): '대구광역시',   # 2023년 대구 편입
}

def resolve_sido(name):
    """예전 명칭(광주광역시/전라남도/강원도/전라북도)을 현재 명칭으로 통일."""
    if pd.isna(name):
        return None
    return SIDO_ALIASES.get(str(name).strip(), str(name).strip())

def canonical_key(sido, sigungu):
    sido = resolve_sido(sido)
    if sido is None:
        return None
    if '세종' in sido:
        return '세종특별자치시'
    return f"{sido} {sigungu}"

def normalize_sgg_for_join(sido, sgg):
    """특례시 하위구 표기('수원시장안구' 또는 시 이름 없이 '북구'만 있는 경우 등)를 상위 시로 되돌린다."""
    if pd.isna(sgg) or str(sgg).strip() == '':
        return None
    sido_r = resolve_sido(sido)
    name = str(sgg).strip()
    if (sido_r, name) in BARE_GU_TO_CITY:
        return BARE_GU_TO_CITY[(sido_r, name)]
    name_nospace = name.replace(' ', '')
    for city in GU_HAVING_CITIES:
        if name_nospace.startswith(city):
            return city
    return name

def broadcast_sigungu_to_eupmyeon(eupmyeon_df, sigungu_score_df, sido_col='sidonm', sgg_col='sggnm', on='sigungu'):
    """시군구 단위 점수를 읍면동 단위 표에 그대로 복제해서 붙인다 (자가용미보유용)."""
    df = eupmyeon_df.copy()
    def make_key(r):
        sido = resolve_sido(r[sido_col])
        sgg_norm = normalize_sgg_for_join(sido, r.get(sgg_col))
        override = LEGACY_SIDO_OVERRIDE.get((sido, sgg_norm))
        if override:
            sido = override
        return canonical_key(sido, sgg_norm if sgg_norm else sido)
    df['_join_key'] = df.apply(make_key, axis=1)
    merged = df.merge(sigungu_score_df, left_on='_join_key', right_on=on, how='left')
    return merged.drop(columns=['_join_key'])

def build_match_key(sido, sgg, dong):
    """읍면동 고유 데이터끼리(의료·버스 등) 조인할 때 쓰는 공통 키."""
    sido_r = resolve_sido(sido)
    sgg_norm = normalize_sgg_for_join(sido_r, sgg) or sido_r
    return f"{sido_r}|{sgg_norm}|{str(dong).strip()}"

def minmax(s):
    """값이 클수록 나쁨(취약)인 지표를 그대로 0~1로."""
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn)

def minmax_invert(s):
    """값이 클수록 좋음(안 취약)인 지표를 뒤집어서 0~1 취약점수로."""
    mn, mx = s.min(), s.max()
    return 1 - (s - mn) / (mx - mn)

In [2]:
print('1) 1인당 자동차등록대수 CSV 업로드')
uploaded = files.upload()
car_path = list(uploaded.keys())[0]

def load_car_percap_sigungu(path):
    df = pd.read_csv(path, encoding='cp949')
    df = df.iloc[1:].reset_index(drop=True)
    df.columns = ['행정구역별', '1인당등록대수']
    df['1인당등록대수'] = pd.to_numeric(df['1인당등록대수'], errors='coerce')

    keys, values = [], []
    cur_sido = None
    skip_sub_gu = False
    for raw, val in zip(df['행정구역별'], df['1인당등록대수']):
        name = str(raw).strip()
        if name == '전남광주통합특별시' and pd.isna(val):
            continue  # 값 없는 표기용 헤더 행 (실제 광주/전라남도는 아래에서 각각 처리됨 -> 자동으로 통합명으로 변환됨)
        if name in NON_REGION:
            continue
        canon = resolve_sido(name)
        if canon in SIDO_NAMES:
            if '세종' in canon and pd.notna(val):
                keys.append('세종특별자치시'); values.append(val)
            cur_sido, skip_sub_gu = canon, False
            continue
        if skip_sub_gu and name.endswith('구'):
            continue
        skip_sub_gu = False
        if name.endswith(('시', '군', '구')):
            if name in GU_HAVING_CITIES:
                skip_sub_gu = True
            if cur_sido:
                keys.append(canonical_key(cur_sido, name)); values.append(val)
            continue

    out = pd.DataFrame({'sigungu': keys, '1인당_자동차등록대수': values})
    return out.groupby('sigungu', as_index=False)['1인당_자동차등록대수'].mean()

car_sigungu = load_car_percap_sigungu(car_path)
car_sigungu['자가용미보유_취약점수'] = minmax_invert(car_sigungu['1인당_자동차등록대수'])
print(car_sigungu.shape, '(광주+전남 통합 후 228개 전후로 나와야 정상)')

1) 1인당 자동차등록대수 CSV 업로드


Saving 1인당_자동차_등록대수_시도_시_군_구__20260923121149.csv to 1인당_자동차_등록대수_시도_시_군_구__20260923121149 (1).csv
(229, 3) (광주+전남 통합 후 228개 전후로 나와야 정상)


In [3]:
print('2) slope_score.csv 업로드 (은규 지형장벽, 기준 테이블)')
uploaded = files.upload()
eupmyeon_master = pd.read_csv(list(uploaded.keys())[0])
print('지형장벽 행 수:', len(eupmyeon_master))

eupmyeon_master = broadcast_sigungu_to_eupmyeon(eupmyeon_master, car_sigungu)
print('자가용미보유 결측:', eupmyeon_master['자가용미보유_취약점수'].isna().sum())
if eupmyeon_master['자가용미보유_취약점수'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['자가용미보유_취약점수'].isna()][['sidonm','sggnm','emdnm']].drop_duplicates())

2) slope_score.csv 업로드 (은규 지형장벽, 기준 테이블)


Saving slope_score.csv to slope_score (1).csv
지형장벽 행 수: 3555
자가용미보유 결측: 77
     sidonm   sggnm emdnm
914   인천광역시     서해구  가정2동
1181  인천광역시     서해구  석남3동
1220    경기도  화성시동탄구  동탄9동
1310    경기도  화성시동탄구  동탄5동
1393  인천광역시     영종구   용유동
...     ...     ...   ...
3471  인천광역시     서해구  청라1동
3473  인천광역시     영종구  운서1동
3482  인천광역시     영종구  영종1동
3498  인천광역시     서해구  청라2동
3542    경기도  화성시만세구   새솔동

[77 rows x 3 columns]


In [5]:
print('2-보완) 고령인구 KOSIS 파일 업로드 (101_DT_1IN1503, 읍면동 단위 재다운로드본)')
uploaded = files.upload()
age_path = list(uploaded.keys())[0]

import xml.etree.ElementTree as ET

def read_kosis_xml_xls(path):
    """KOSIS가 .xls 확장자로 내려주지만 실제로는 XML(엑셀 2003 XML) 형식인 파일을 파싱."""
    with open(path, 'rb') as f:
        raw = f.read()
    text = raw.decode('euc-kr', errors='replace').lstrip()
    text = text.replace('encoding="EUC-KR"', 'encoding="UTF-8"')
    idx = text.find('메타정보')
    if idx != -1:
        idx2 = text.rfind('<Worksheet', 0, idx)
        text = text[:idx2] + '</Workbook>'
    root = ET.fromstring(text)
    ns = '{urn:schemas-microsoft-com:office:spreadsheet}'
    rows_out = []
    for ws in root.findall(f'{ns}Worksheet'):
        table = ws.find(f'{ns}Table')
        if table is None:
            continue
        for row in table.findall(f'{ns}Row'):
            cells = []
            for cell in row.findall(f'{ns}Cell'):
                data = cell.find(f'{ns}Data')
                cells.append(data.text if data is not None else None)
            rows_out.append(cells)
    return rows_out

def load_age_eupmyeondong(path):
    raw_rows = read_kosis_xml_xls(path)
    data_rows = [r for r in raw_rows[2:] if r and len(r) >= 5]
    df = pd.DataFrame(data_rows, columns=['행정구역_raw', '연령별', '항목', '단위', '2025'])
    df['2025'] = pd.to_numeric(df['2025'], errors='coerce')

    names = df['행정구역_raw'].astype(str).str.strip()
    block_id = (names != names.shift()).cumsum()        # 연속 동일 라벨을 하나의 블록으로
    block_labels = names.groupby(block_id).first()

    cur_sido, cur_sgg, cur_sgg_is_gu_city = None, None, False
    block_sido, block_sgg, block_dong = {}, {}, {}

    for bid, name in block_labels.items():
        if name in NON_REGION:
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        canon = resolve_sido(name)
        if canon in SIDO_NAMES:
            cur_sido, cur_sgg, cur_sgg_is_gu_city = canon, None, False
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        if name.endswith('구') and cur_sgg_is_gu_city:
            # 특례시 하위 자치구(예: 수원시 장안구) -> 구조 표시일 뿐, 실제 단위 아님
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        if name.endswith(('시', '군', '구')):
            cur_sgg = name
            cur_sgg_is_gu_city = name in GU_HAVING_CITIES
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        # 읍면동 레벨
        block_sido[bid] = cur_sido
        block_sgg[bid] = cur_sgg
        block_dong[bid] = name

    df['_block'] = block_id
    df['sido'] = df['_block'].map(block_sido)
    df['sgg'] = df['_block'].map(block_sgg)
    df['dong'] = df['_block'].map(block_dong)

    valid = df[df['dong'].notna()].copy()
    total = valid[valid['연령별'] == '합계'].groupby(['sido', 'sgg', 'dong'])['2025'].sum().reset_index(name='전체인구')
    elderly = valid[valid['연령별'] == '65세이상'].groupby(['sido', 'sgg', 'dong'])['2025'].sum().reset_index(name='고령인구')
    out = total.merge(elderly, on=['sido', 'sgg', 'dong'], how='left')
    out['고령인구비율'] = out['고령인구'] / out['전체인구']
    return out

age_eupmyeon = load_age_eupmyeondong(age_path)
print(age_eupmyeon.shape, '(3,555~3,558개 근처여야 정상)')
print('전체인구 합계 (전국 51,817,499명과 비교):', age_eupmyeon['전체인구'].sum())

age_eupmyeon['_match_key'] = age_eupmyeon.apply(lambda r: build_match_key(r['sido'], r['sgg'], r['dong']), axis=1)
eupmyeon_master['_match_key'] = eupmyeon_master.apply(lambda r: build_match_key(r['sidonm'], r['sggnm'], r['emdnm']), axis=1)

eupmyeon_master = eupmyeon_master.merge(
    age_eupmyeon[['_match_key', '고령인구비율']], on='_match_key', how='left'
)
eupmyeon_master = eupmyeon_master.drop(columns=['_match_key'])
eupmyeon_master['고령인구비율_취약점수'] = minmax(eupmyeon_master['고령인구비율'])  # 비율이 높을수록 취약

print('고령인구비율 결측:', eupmyeon_master['고령인구비율'].isna().sum())
if eupmyeon_master['고령인구비율'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['고령인구비율'].isna()][['sidonm', 'sggnm', 'emdnm']].drop_duplicates().head(20))

2-보완) 고령인구 KOSIS 파일 업로드 (101_DT_1IN1503, 읍면동 단위 재다운로드본)


Saving 101_DT_1IN1503_20261004030052.xls to 101_DT_1IN1503_20261004030052 (1).xls
(3557, 6) (3,555~3,558개 근처여야 정상)
전체인구 합계 (전국 51,817,499명과 비교): 51817499.0
고령인구비율 결측: 85
     sidonm   sggnm  emdnm
128     경기도  안양시만안구   병목안동
847   부산광역시    부산진구   당감1동
886     경기도  안양시만안구    명학동
914   인천광역시     서해구   가정2동
1088    경기도  용인시처인구    양지읍
1181  인천광역시     서해구   석남3동
1220    경기도  화성시동탄구   동탄9동
1310    경기도  화성시동탄구   동탄5동
1393  인천광역시     영종구    용유동
1510  인천광역시     서해구   가정3동
1586    경기도  화성시동탄구   동탄8동
1636    경기도  화성시효행구    봉담읍
1655  인천광역시     검단구  불로대곡동
1670  인천광역시     검단구   아라2동
1671  인천광역시     서해구    연희동
1682    경기도  화성시효행구    비봉면
1713  인천광역시     검단구    검단동
1748  인천광역시     서해구   가좌2동
1749    경기도  화성시동탄구   동탄7동
1891    경기도  화성시효행구    매송면


In [6]:
print('3) medical_access_score.csv 업로드 (은규 의료접근성)')
uploaded = files.upload()
medical_df = pd.read_csv(list(uploaded.keys())[0])

if '의료_취약점수' not in medical_df.columns:
    W1, W2 = 0.5, 0.5   # TODO: 은규가 실제로 쓴 1차/2차 가중치로 교체
    medical_df['의료_취약점수'] = (
        W1 * medical_df['1차의료_취약점수'] + W2 * medical_df['2차의료_취약점수']
    )

eupmyeon_master = eupmyeon_master.merge(
    medical_df[['읍면_key', '의료_취약점수']], on='읍면_key', how='left'
)
eupmyeon_master['의료_취약점수'] = eupmyeon_master['의료_취약점수'].fillna(0)  # 동 지역: 마을조사표 대상 아님 -> 0 가정, 은규 확인 필요

3) medical_access_score.csv 업로드 (은규 의료접근성)


Saving medical_access_score.csv to medical_access_score (1).csv


In [7]:
print('4-1) stop_supply_by_dong.csv 업로드 (가연 버스정류장·인구대비, 2차 수정본)')
uploaded = files.upload()
stop_df = pd.read_csv(list(uploaded.keys())[0])

print('4-2) dong_supply.csv 업로드 (가연 배차간격, 2차 수정본)')
uploaded = files.upload()
headway_df = pd.read_csv(list(uploaded.keys())[0])

stop_df['버스정류장_취약점수'] = minmax_invert(stop_df['per10k'])          # 적을수록 취약
headway_df['배차간격_취약점수'] = minmax(headway_df['headway_min'])        # 길수록 취약

stop_df['_match_key'] = stop_df.apply(lambda r: build_match_key(r['province'], r['sigungu'], r['dong']), axis=1)
headway_df['_match_key'] = headway_df.apply(lambda r: build_match_key(r['province'], r['sigungu'], r['dong']), axis=1)
eupmyeon_master['_match_key'] = eupmyeon_master.apply(lambda r: build_match_key(r['sidonm'], r['sggnm'], r['emdnm']), axis=1)

eupmyeon_master = eupmyeon_master.merge(
    stop_df[['_match_key', '버스정류장_취약점수']], on='_match_key', how='left'
).merge(
    headway_df[['_match_key', '배차간격_취약점수']], on='_match_key', how='left'
)
eupmyeon_master = eupmyeon_master.drop(columns=['_match_key'])

print('버스정류장_취약점수 결측 (대체 전):', eupmyeon_master['버스정류장_취약점수'].isna().sum())
print('배차간격_취약점수 결측 (대체 전):', eupmyeon_master['배차간격_취약점수'].isna().sum())

# 결측 여부를 먼저 플래그로 남겨둠 (보고서용)
eupmyeon_master['버스정류장_취약점수_대체여부'] = eupmyeon_master['버스정류장_취약점수'].isna()
eupmyeon_master['배차간격_취약점수_대체여부'] = eupmyeon_master['배차간격_취약점수'].isna()

def fill_by_region_mean(df, col, sgg_col='sggnm', sido_col='sidonm'):
    sgg_mean = df.groupby(sgg_col)[col].transform('mean')
    df[col] = df[col].fillna(sgg_mean)
    sido_mean = df.groupby(sido_col)[col].transform('mean')
    df[col] = df[col].fillna(sido_mean)
    return df

eupmyeon_master = fill_by_region_mean(eupmyeon_master, '버스정류장_취약점수')
eupmyeon_master = fill_by_region_mean(eupmyeon_master, '배차간격_취약점수')

print('버스정류장_취약점수 결측 (대체 후, 0이어야 정상):', eupmyeon_master['버스정류장_취약점수'].isna().sum())
print('배차간격_취약점수 결측 (대체 후, 0이어야 정상):', eupmyeon_master['배차간격_취약점수'].isna().sum())
print('잠정 대체된 읍면동 수 — 버스정류장:', eupmyeon_master['버스정류장_취약점수_대체여부'].sum(),
      ' / 배차간격:', eupmyeon_master['배차간격_취약점수_대체여부'].sum())

4-1) stop_supply_by_dong.csv 업로드 (가연 버스정류장·인구대비, 2차 수정본)


Saving stop_supply_by_dong.csv to stop_supply_by_dong (1).csv
4-2) dong_supply.csv 업로드 (가연 배차간격, 2차 수정본)


Saving dong_supply.csv to dong_supply (1).csv
버스정류장_취약점수 결측 (대체 전): 144
배차간격_취약점수 결측 (대체 전): 13
버스정류장_취약점수 결측 (대체 후, 0이어야 정상): 0
배차간격_취약점수 결측 (대체 후, 0이어야 정상): 0
잠정 대체된 읍면동 수 — 버스정류장: 144  / 배차간격: 13


In [8]:
# 공급공백(완성되면) 나오는 대로 여기 추가
final_vars = ['지형장벽_취약점수', '자가용미보유_취약점수', '의료_취약점수',
              '버스정류장_취약점수', '배차간격_취약점수', '고령인구비율_취약점수']

# 엔트로피 계산 전 필수 점검: 여기서 하나라도 0이 아니면, entropy_weights의
# dropna()가 그 행들을 통째로 날려서 가중치 자체가 왜곡될 수 있음 -> 반드시 확인
print('엔트로피 계산 직전 변수별 결측 개수:')
print(eupmyeon_master[final_vars].isna().sum())

def entropy_weights(df, cols, eps=1e-6):
    X = df[cols].copy().dropna()
    X = X * (1 - 2*eps) + eps
    P = X / X.sum(axis=0)
    n = len(X)
    E = -(P * np.log(P)).sum(axis=0) / np.log(n)
    D = 1 - E
    W = D / D.sum()
    return W

weights = entropy_weights(eupmyeon_master, final_vars)
print(weights)

eupmyeon_master['이동격차지수'] = sum(eupmyeon_master[v].fillna(0) * weights[v] for v in final_vars)

final_ranked = eupmyeon_master[~eupmyeon_master['sidonm'].str.contains('서울')].copy()
final_ranked = final_ranked.sort_values('이동격차지수', ascending=False)
print(final_ranked[['읍면_key'] + final_vars + ['이동격차지수']].head(20))

엔트로피 계산 직전 변수별 결측 개수:
지형장벽_취약점수       0
자가용미보유_취약점수    77
의료_취약점수         0
버스정류장_취약점수      0
배차간격_취약점수       0
고령인구비율_취약점수    85
dtype: int64
지형장벽_취약점수      0.099255
자가용미보유_취약점수    0.003218
의료_취약점수        0.588743
버스정류장_취약점수     0.006348
배차간격_취약점수      0.247658
고령인구비율_취약점수    0.054777
dtype: float64
               읍면_key  지형장벽_취약점수  자가용미보유_취약점수   의료_취약점수  버스정류장_취약점수  \
51      경상북도 울진군 금강송면   0.802846     0.878788  0.881121    0.692876   
62       충청북도 영동군 용화면   0.793058     0.878788  0.777656    0.475824   
61       경상남도 함안군 여항면   0.795086     0.666667  0.766428    0.640533   
30    강원특별자치도 삼척시 하장면   0.827959     0.878788  0.759669    0.545151   
994     인천광역시 옹진군 연평면   0.454124     0.333333  0.910529    0.962967   
2189       경기도 이천시 율면   0.210121     0.878788  0.913146    0.977705   
322      충청북도 영동군 양강면   0.633390     0.878788  0.779606    0.680591   
70    강원특별자치도 춘천시 북산면   0.779212     0.909091  0.785818    0.620696   
1075  전북특별자치도 군산시 옥도면   0.436033     0.878788  0.954533    

In [9]:
equal_w = {v: 1/len(final_vars) for v in final_vars}
eupmyeon_master['이동격차지수_동일가중'] = sum(
    eupmyeon_master[v].fillna(0) * equal_w[v] for v in final_vars
)

comparison = eupmyeon_master[['읍면_key']].copy()
comparison['순위_엔트로피'] = eupmyeon_master['이동격차지수'].rank(ascending=False)
comparison['순위_동일가중'] = eupmyeon_master['이동격차지수_동일가중'].rank(ascending=False)
comparison['순위차이'] = (comparison['순위_엔트로피'] - comparison['순위_동일가중']).abs()

print('상위 20위 안에서 순위가 가장 많이 바뀐 지역:')
print(comparison.sort_values('순위차이', ascending=False).head(20))

상위 20위 안에서 순위가 가장 많이 바뀐 지역:
                 읍면_key  순위_엔트로피  순위_동일가중    순위차이
2305  전남광주통합특별시 신안군 장산면   1145.0   3453.0  2308.0
2084  전남광주통합특별시 신안군 지도읍   1308.0   3468.0  2160.0
1382  전남광주통합특별시 신안군 암태면   1345.0   3470.0  2125.0
2224  전남광주통합특별시 신안군 비금면   1350.0   3471.0  2121.0
1379  전남광주통합특별시 신안군 도초면   1342.0   3461.0  2119.0
1854  전남광주통합특별시 신안군 임자면   1362.0   3472.0  2110.0
1690  전남광주통합특별시 신안군 신의면   1401.0   3474.0  2073.0
2784  전남광주통합특별시 신안군 증도면   1420.0   3473.0  2053.0
1958  전남광주통합특별시 신안군 자은면   1429.0   3478.0  2049.0
2547  전남광주통합특별시 신안군 안좌면   1451.0   3477.0  2026.0
1871  전남광주통합특별시 신안군 하의면   1463.0   3467.0  2004.0
3407  전남광주통합특별시 해남군 산이면   1489.0   3452.0  1963.0
2497  전남광주통합특별시 해남군 황산면   1326.0   3286.0  1960.0
1777  전남광주통합특별시 해남군 해남읍   1465.0   3372.0  1907.0
1856  전남광주통합특별시 해남군 화산면   1340.0   3244.0  1904.0
2633  전남광주통합특별시 신안군 팔금면   1572.0   3475.0  1903.0
2174  전남광주통합특별시 신안군 압해읍   1607.0   3476.0  1869.0
1927  전남광주통합특별시 해남군 마산면   1444.0   3208.0  1764.0
3330    전북특별자치도 김제시 신풍